# User Migration — up_users → users + user_roles

**Migration 2 of the run order** — run after `geo_migration.ipynb`.

Migrates legacy Strapi users (users-permissions plugin) into the new `users` table and assigns each a `user_roles` row from their legacy `user_type`.

**Read [docs/migrations/user-migration.md](../docs/migrations/user-migration.md) first** — full field mapping, dropped fields, and the password caveat.

**Prerequisites (in order):** schema migrated → `python scripts/seed.py` (user_types must exist) → **geo_migration notebook** (country lookups). Idempotent — safe to re-run.

⚠️ `password` is a private field — the Strapi REST API never returns it. `password_hash` stays NULL here; recover hashes via a direct dump of the old DB or force a password reset (see doc).

In [ ]:
import os, json
from pathlib import Path

import requests
import psycopg
from psycopg.types.json import Json
from dotenv import load_dotenv

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
load_dotenv(ROOT / ".env")

CMS_BASE_URL = os.environ["CMS_BASE_URL"].rstrip("/")
HEADERS = {"Authorization": f"Bearer {os.environ['CMS_API_TOKEN']}"}
DATABASE_URL = os.environ["DATABASE_URL"]

conn = psycopg.connect(DATABASE_URL)
print("connected to:", DATABASE_URL.rsplit("/", 1)[-1])

## 1. Fetch legacy users

The users-permissions endpoint `/api/users` returns **flat** objects (no `data/attributes` wrapper) and paginates with `start`/`limit`.

In [ ]:
def fetch_users():
    users, start, limit = [], 0, 100
    while True:
        r = requests.get(
            f"{CMS_BASE_URL}/api/users",
            headers=HEADERS,
            params={"start": start, "limit": limit,
                    "populate": "user_type,country,company"},
            timeout=60,
        )
        r.raise_for_status()
        batch = r.json()
        users.extend(batch)
        if len(batch) < limit:
            return users
        start += limit


legacy_users = fetch_users()
print(f"fetched {len(legacy_users)} legacy users")

## 2. Transform + upsert into `users`

Mapping highlights (full table in the doc):
`confirmed→email_verified`, `blocked→is_blocked`, `provider→auth_provider`, `fbId→provider_id`, `isLoyaltyMember→tier` (star_life/free), `profilePicURL→media row→profile_pic_id`. Legacy free-text `city` cannot be resolved to a `city_id` automatically — collected for manual review.

In [ ]:
def split_name(u):
    first, last = u.get("firstName"), u.get("lastName")
    if not first and u.get("fullName"):
        parts = u["fullName"].strip().split(" ", 1)
        first = parts[0]
        last = last or (parts[1] if len(parts) > 1 else None)
    return first, last


skipped, city_review = [], []
with conn.cursor() as cur:
    for u in legacy_users:
        email = (u.get("email") or "").strip().lower()
        username = (u.get("username") or "").strip() or (email.split("@")[0] if email else None)
        if not email or not username:
            skipped.append((u["id"], u.get("username"), u.get("email")))
            continue

        first, last = split_name(u)

        # profilePicURL → media row (reused if already inserted)
        profile_pic_id = None
        if u.get("profilePicURL"):
            cur.execute("SELECT id FROM media WHERE url = %s", (u["profilePicURL"],))
            row = cur.fetchone()
            if not row:
                cur.execute("INSERT INTO media (url) VALUES (%s) RETURNING id", (u["profilePicURL"],))
                row = cur.fetchone()
            profile_pic_id = row[0]

        country = u.get("country") or {}
        country_name = (country.get("name") or "").strip() or None

        cur.execute(
            """
            INSERT INTO users (username, email, first_name, last_name,
                               auth_provider, provider_id, email_verified, is_blocked,
                               profile_pic_id, bio, social, seo, tracking,
                               is_featured, priority, tier,
                               country_id)
            VALUES (%s, %s, %s, %s,
                    %s, %s, %s, %s,
                    %s, %s, %s, %s, %s,
                    %s, %s, %s,
                    (SELECT id FROM countries WHERE name = %s))
            ON CONFLICT (email) DO UPDATE
            SET username = EXCLUDED.username,
                first_name = EXCLUDED.first_name,
                last_name = EXCLUDED.last_name,
                email_verified = EXCLUDED.email_verified,
                is_blocked = EXCLUDED.is_blocked,
                tier = EXCLUDED.tier
            """,
            (
                username, email, first, last,
                u.get("provider"), u.get("fbId"), bool(u.get("confirmed")), bool(u.get("blocked")),
                profile_pic_id, u.get("bio"),
                Json(u["social"]) if u.get("social") else None,
                Json(u["seo"]) if u.get("seo") else None,
                Json(u["tracking"]) if u.get("tracking") else None,
                bool(u.get("isFeatured")), u.get("priority") or 0,
                "star_life" if u.get("isLoyaltyMember") else "free",
                country_name,
            ),
        )

        if u.get("city"):
            city_review.append((email, u["city"]))

conn.commit()
print(f"upserted users; skipped (no email/username): {len(skipped)}")
print(f"MANUAL REVIEW — free-text city to resolve to city_id: {len(city_review)}")
city_review[:10]

## 3. Assign roles — legacy `user_type` → `user_roles`

Legacy slugs are matched against the seeded `user_types`; users without a legacy user_type get the default **member** role. (Partner ↔ Company linking via `user_roles.company_id` is left for the company migration.)

In [ ]:
unmatched_types = set()
with conn.cursor() as cur:
    for u in legacy_users:
        email = (u.get("email") or "").strip().lower()
        if not email:
            continue
        ut = u.get("user_type") or {}
        type_slug = (ut.get("slug") or "member").strip().lower()

        cur.execute("SELECT id FROM user_types WHERE slug = %s", (type_slug,))
        row = cur.fetchone()
        if not row:
            unmatched_types.add(type_slug)
            cur.execute("SELECT id FROM user_types WHERE is_default")
            row = cur.fetchone()

        cur.execute(
            """
            INSERT INTO user_roles (user_id, user_type_id, status)
            SELECT us.id, %s, 'active' FROM users us
            WHERE us.email = %s
              AND NOT EXISTS (
                    SELECT 1 FROM user_roles ur
                    WHERE ur.user_id = us.id AND ur.user_type_id = %s)
            """,
            (row[0], email, row[0]),
        )
conn.commit()
print(f"roles assigned; legacy user_type slugs with no match (fell back to member): {unmatched_types}")

## 4. Verify

In [ ]:
with conn.cursor() as cur:
    cur.execute("SELECT COUNT(*) FROM users")
    print("users     :", cur.fetchone()[0])
    cur.execute(
        """SELECT ut.slug, COUNT(*) FROM user_roles ur
           JOIN user_types ut ON ut.id = ur.user_type_id
           GROUP BY ut.slug ORDER BY 2 DESC"""
    )
    for slug, n in cur.fetchall():
        print(f"role {slug:12}: {n}")
    cur.execute("SELECT COUNT(*) FROM users WHERE password_hash IS NULL")
    print("users without password_hash (expected — see doc):", cur.fetchone()[0])
conn.close()